In [3]:
import pandas as pd
import plotly.express as px
from rich.progress import track
import pathlib
import cogent3

paths = {
    "human_chimp": pathlib.Path('~/source/madb_data/thesis_data/human_chimp/ensembl_alignments/').expanduser(),
    "human_gorilla": pathlib.Path('~/source/madb_data/thesis_data/human_gorilla/ensembl_alignments/').expanduser(),
    "human_macaque": pathlib.Path('~/source/madb_data/thesis_data/human_macaque/ensembl_alignments/').expanduser()
}

pdists = []

for pair, path in paths.items():
    for fn in track(list(path.glob('*.fa')), description=f"Processing {pair}"):
        aln = cogent3.load_aligned_seqs(fn, moltype='dna')
        pdist = aln.distance_matrix(calc='pdist')
        pdists.append((pair, pdist[0, 1]))

df = pd.DataFrame(pdists, columns=["Pair", "Distance"])

# Rename comparison groups
df["Pair"] = df["Pair"].map({
    "human_chimp": "Chimpanzee",
    "human_gorilla": "Gorilla",
    "human_macaque": "Macaque"
})

# Create violin plot
fig = px.violin(
    df,
    x="Distance",
    y="Pair",
    orientation='h',
    labels={"Distance": "pdist", "Pair": ""},
    color="Pair",
    color_discrete_map={
        "Chimpanzee": "blue",
        "Gorilla": "green",
        "Macaque": "red"
    }
)

fig.update_layout(
    violinmode='overlay',
    yaxis=dict(categoryorder="array", categoryarray=["Macaque", "Gorilla", "Chimpanzee"]),
    margin=dict(l=60, r=20, t=40, b=40),
    height=280
)

fig.update_traces(width=0.8, marker=dict(size=3), showlegend=False)

# Save figure
figures_dir = pathlib.Path("..") / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(figures_dir / "pdist_pairs.png")
fig.show()


Output()

Output()

Output()